# Week 7 Day 3: Introduction to Agentic AI with CrewAI

---

# Introduction

In the previous lectures, we built AI applications using a **single Large Language Model (LLM)**.

Although a single LLM is powerful, many real-world problems are better solved by dividing the work among multiple AI agents.

This concept is called **Agentic AI**.

Instead of one AI performing every task, multiple AI agents collaborate, where each agent specializes in a particular job.

In this notebook, we will learn:

- What is Agentic AI?
- What is CrewAI?
- Agent
- Task
- Crew
- Process
- Memory
- Running multiple AI agents using Google Gemini

---

# What is CrewAI?

CrewAI is an open-source Python framework for building **multi-agent AI applications**.

Instead of giving all work to one LLM, we create multiple specialized AI agents.

Each agent has:

- A Role
- A Goal
- A Backstory
- A Task

The agents collaborate together to solve a larger problem.

---

# Project: AI News Research Team

## Scenario

Suppose we want an AI team to write a blog on the topic:

> **"Latest Advancements in Artificial Intelligence"**

Instead of asking a single Large Language Model (LLM) to perform the entire task, we divide the work among multiple specialized AI agents.

```
                 User
                   │
                   ▼
            Crew (Manager)
                   │
     ┌─────────────┴──────────────┐
     │                            │
     ▼                            ▼
Research Agent              Writer Agent
     │                            │
     ▼                            ▼
Collect Information        Write Blog
             \              /
              \            /
               └─────┬────┘
                     ▼
               Final Output
```

### Workflow

1. The **User** provides a topic.
2. The **Crew** coordinates the entire workflow.
3. The **Research Agent** gathers relevant information about the topic.
4. The **Writer Agent** uses the research to write a beginner-friendly blog.
5. The Crew returns the **final blog** to the user.

This simple example demonstrates the core idea of **Agentic AI**, where multiple specialized AI agents collaborate to solve a problem more effectively than a single AI model handling every task.

---

# Example Problem

Suppose we want to write a blog on **Artificial Intelligence**.

Instead of asking one AI model to perform everything,

we divide the work.

```
User
   │
   ▼
Research Agent
   │
Research Notes
   │
   ▼
Writer Agent
   │
   ▼
Final Blog
```

Each agent performs only the task it specializes in.

---

# CrewAI Components

CrewAI mainly consists of four components.

## 1. Agent

An Agent is an AI worker.

Examples

- Research Agent
- Blog Writer
- Travel Planner
- Financial Analyst
- Data Scientist

Every agent has

- Role
- Goal
- Backstory
- LLM

---

## 2. Task

A Task represents the work assigned to an agent.

Examples

- Research Artificial Intelligence
- Write Blog
- Analyse Sales Data
- Create Travel Plan

---

## 3. Crew

A Crew is simply a collection of multiple agents working together.

---

## 4. Process

The process determines how tasks are executed.

CrewAI currently supports multiple execution strategies.

In this notebook we will use

**Sequential Process**

which executes tasks one after another.

---

# Install Required Libraries

```python
!pip install -q crewai python-dotenv
```

---

# Import Libraries

```python
from crewai import Agent, Task, Crew, Process, LLM
from dotenv import load_dotenv
import os
```

---

# Load Environment Variables

Store your Gemini API Key inside a `.env` file.

```
GEMINI_API_KEY=YOUR_GEMINI_API_KEY
```

Now load the environment variables.

```python
load_dotenv()
```

---

# Configure Google Gemini

CrewAI supports multiple Large Language Models.

Here we will use **Google Gemini 2.5 Flash**.

```python
llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.7
)
```

---

# Previous Chat History

Suppose our chatbot already had a previous conversation with the user.

We can pass that conversation to the Crew.

```python
chat_history = """
User: Hi, I want to learn Artificial Intelligence.

Assistant: Great! Artificial Intelligence is a branch of computer science
that enables machines to perform tasks that normally require human intelligence.

User: Can you also explain current trends?
"""
```

---

# Creating the Research Agent

The first agent is responsible for researching the topic.

```python
research_agent = Agent(
    role="AI Researcher",
    goal="Find useful information about the given topic.",
    backstory=(
        "You are an expert AI researcher who gathers accurate and "
        "well-structured information."
    ),
    llm=llm,
    verbose=True
)
```

---

# Creating the Writer Agent

The second agent converts the research into a simple blog.

```python
writer_agent = Agent(
    role="Technical Blog Writer",
    goal="Write beginner-friendly technical blogs.",
    backstory=(
        "You convert technical research into simple and engaging articles."
    ),
    llm=llm,
    verbose=True
)
```

---

# Task 1

The first task asks the Research Agent to collect information.

```python
research_task = Task(
    description="""
Previous Conversation:

{chat_history}

Research the topic:

Artificial Intelligence

Collect:

- Definition
- Latest Trends
- Applications
- Future Scope

Use the previous conversation whenever relevant.

Return detailed research notes.
""",
    expected_output="Detailed research notes",
    agent=research_agent
)
```

---

# Task 2

The second task asks the Writer Agent to convert the research into a beginner-friendly blog.

```python
writing_task = Task(
    description="""
Previous Conversation:

{chat_history}

Using the research notes from the previous task,
write a beginner-friendly blog.

Include:

- Introduction
- Latest Trends
- Applications
- Future Scope
- Conclusion

If the user's previous conversation indicates any preferences,
adapt the writing accordingly.
""",
    expected_output="Complete blog article",
    agent=writer_agent
)
```

---

# Creating the Crew

Now we combine the agents and tasks into one Crew.

We also enable memory.

```python
crew = Crew(
    agents=[
        research_agent,
        writer_agent
    ],
    tasks=[
        research_task,
        writing_task
    ],
    process=Process.sequential,
    memory=True,
    verbose=True
)
```

---

# Running the Crew

The Crew is executed using `kickoff()`.

We also pass the previous chat history as input.

```python
result = crew.kickoff(
    inputs={
        "chat_history": chat_history
    }
)
```

---

# Display the Final Output

```python
print("\n================ FINAL OUTPUT ================\n")
print(result)
```

---

# Understanding the Workflow

The complete execution pipeline is shown below.

```
                User
                  │
                  ▼
        Previous Chat History
                  │
                  ▼
          Research Agent
                  │
          Research Notes
                  │
                  ▼
           Writer Agent
                  │
          Beginner Blog
                  │
                  ▼
            Final Output
```

Notice that each agent performs only the task assigned to it.

---

# Understanding `memory=True`

```python
memory=True
```

This enables CrewAI's internal memory.

The information generated by previous tasks becomes available to later tasks during the Crew execution.

Example

```
Research Agent
       │
       ▼
Research Notes
       │
       ▼
Writer Agent
```

The Writer Agent automatically receives the research generated by the Research Agent.

**Important**

CrewAI memory is available **only during the current Crew execution**.

It does **not** automatically remember conversations after the Python program terminates.

---

# Understanding `{chat_history}`

Inside our task description we used

```python
{chat_history}
```

During execution,

```python
crew.kickoff(
    inputs={
        "chat_history": chat_history
    }
)
```

CrewAI replaces

```python
{chat_history}
```

with the actual conversation.

This mechanism is called **Template Variable Substitution**.

---

# Does This Program Perform Web Search?

**No.**

Although the agent is called a **Research Agent**, it does **not** search the internet.

The Research Agent only uses the Gemini LLM's internal knowledge.

Current workflow

```
Task
   │
   ▼
Gemini LLM
   │
   ▼
Generated Response
```

No external websites are accessed.

---

# How Can We Enable Web Search?

To perform real internet searches, we need to provide a **Tool** to the agent.

For example,

```python
from crewai_tools import SerperDevTool

search_tool = SerperDevTool()
```

Then attach the tool to the agent.

```python
research_agent = Agent(
    ...
    tools=[search_tool]
)
```

The workflow then becomes

```
Task
   │
   ▼
Research Agent
   │
   ├────────► Google Search Tool
   │
   ▼
Gemini LLM
   │
   ▼
Final Answer
```

We will study Tools in the next lecture.

---

# Advantages of CrewAI

- Easy to create multiple AI agents
- Clear separation of responsibilities
- Reusable agents
- Supports multiple LLM providers
- Supports Tools
- Supports Memory
- Supports Sequential and Hierarchical workflows
- Easy integration with external APIs

---

# Summary

In this notebook, we learned

- Introduction to Agentic AI
- Introduction to CrewAI
- Agent
- Task
- Crew
- Process
- Google Gemini Integration
- Passing Previous Chat History
- Crew Memory
- Running a Multi-Agent Application
- Difference between LLM Knowledge and Web Search

CrewAI enables developers to build intelligent applications where multiple specialized AI agents collaborate to solve complex problems efficiently.